# final kaggle run

Run this notebook top to bottom in a fresh Kaggle GPU session with Internet enabled. It runs the one controlled training-length revision, the grouped data-size ablation, then freezes the settings before fetching or reading the internal test and external transfer complaints.

The two-epoch revision is the planned final candidate. Keep the validation comparison and the ablation outputs even if the revision is worse. Do not change prompts, labels, data, decoding, or evaluator after the freeze cell.

In [ ]:
!pip -q install 'transformers==5.10.1' 'peft==0.20.0' 'trl==0.29.0' bitsandbytes accelerate datasets 'mlflow==3.15.1' scikit-learn

import hashlib
import json
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

REPO_REF = '3e91316'
RAW_BASE = f'https://raw.githubusercontent.com/goyashek/civic-grievance-structurer/{REPO_REF}'
ROOT = Path('/kaggle/working/civicstruct')
ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_FILES = (
    'src/evaluate.py',
    'src/schema.py',
    'data/surface_variants.jsonl',
    'data/public_training_examples.jsonl',
    'data/dataset_manifest.json',
    'data/validation_results/qlora_validation_predictions.json',
)

def fetch(relative_path):
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(f'{RAW_BASE}/{relative_path}', timeout=60) as response:
            destination.write_bytes(response.read())
    except URLError as exc:
        raise RuntimeError('Turn on Kaggle Internet to fetch the frozen project files.') from exc

for relative_path in TRAIN_FILES:
    fetch(relative_path)
print({'source': RAW_BASE, 'fetched': list(TRAIN_FILES), 'test_loaded': False})

In [ ]:
import gc
import enum
import math
import os
import platform
import random
import shutil
import sys
import time
from collections import defaultdict
from importlib.metadata import version

sys.path.insert(0, str(ROOT))
import mlflow
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from src.evaluate import evaluate_outputs
from src.schema import validate_gold

SEED = 42
MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'
MODEL_REVISION = 'a07cc9a04f16550a088caea529712d1d335b0ac1'
REVISION_EPOCHS = 2
MAX_NEW_TOKENS = 256
MAX_LENGTH = 768
DEMO_COUNT = 3
BATCH_SIZE = 4
MODEL_DTYPE = torch.float16
OUTPUT = Path('/kaggle/working/civicstruct_final_output')
OUTPUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a Kaggle GPU runtime before running this notebook.'
torch.cuda.manual_seed_all(SEED)
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri((OUTPUT / 'mlruns').as_uri())
mlflow.set_experiment('civicstruct-final')

def json_default(value):
    if isinstance(value, set):
        return sorted(value, key=str)
    if isinstance(value, (Path, os.PathLike)):
        return os.fspath(value)
    if isinstance(value, torch.dtype):
        return str(value)
    if isinstance(value, enum.Enum):
        return value.value
    if hasattr(value, 'item'):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass
    if hasattr(value, 'tolist'):
        return value.tolist()
    return str(value)

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=json_default), encoding='utf-8')

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print({'device': torch.cuda.get_device_name(0), 'python': platform.python_version(), 'torch': torch.__version__})

In [ ]:
manifest = json.loads((ROOT / 'data/dataset_manifest.json').read_text(encoding='utf-8'))
for relative_path in ('data/surface_variants.jsonl', 'data/public_training_examples.jsonl'):
    assert sha256(ROOT / relative_path) == manifest['sha256'][relative_path], relative_path

surfaces = load_jsonl(ROOT / 'data/surface_variants.jsonl')
public_training = load_jsonl(ROOT / 'data/public_training_examples.jsonl')
controlled_training = [row for row in surfaces if row['split'] == 'train']
validation = [row for row in surfaces if row['split'] == 'validation']
training = [dict(row) for row in controlled_training] + [dict(row) for row in public_training]
for row in training + validation:
    row.setdefault('surface_id', row['case_id'])
assert len(controlled_training) == 120
assert len(public_training) == 40
assert len(training) == 160
assert len(validation) == 50
assert len({row['surface_id'] for row in validation}) == len(validation)
assert len({row['case_id'] for row in training}) == len(training)
assert not {row['case_id'] for row in training} & {row['case_id'] for row in validation}
print({'training': len(training), 'validation': len(validation), 'test_loaded': False, 'dataset_version': manifest['dataset_version']})

In [ ]:
DOMAINS = ['public_transport', 'water_supply', 'sanitation_and_waste', 'roads_and_streetlights', 'electricity', 'welfare_or_document_service', 'other']
ISSUES = ['delay_or_non_arrival', 'service_outage_or_non_delivery', 'damaged_infrastructure', 'overcharging_or_payment_problem', 'record_or_document_error', 'staff_conduct', 'safety_or_health_hazard', 'other']
URGENCY = ['routine', 'time_sensitive', 'safety_critical']
MISSING = ['exact_location', 'date_or_time', 'service_identifier', 'transaction_or_reference_id', 'amount', 'supporting_evidence', 'affected_person_or_group', 'none']
SYSTEM_PROMPT = (
    'Structure one public-service complaint as exactly one JSON object. Use these fields in this order: '
    'service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, urgency, missing_information, formal_summary. '
    f'Allowed service_domain values: {DOMAINS}. Allowed issue_type values: {ISSUES}. '
    f'Allowed urgency values: {URGENCY}. Allowed missing_information values: {MISSING}. '
    'Use null for absent scalar facts. Missing information must be a non-empty ordered list. Use the none label only when no important detail is missing. '
    'Do not guess facts. The formal summary must be one neutral sentence. Return no reasoning, markdown, or commentary.'
)

def messages_for(complaint, demos=()):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for demo in demos:
        messages.extend([
            {'role': 'user', 'content': demo['complaint']},
            {'role': 'assistant', 'content': json.dumps(demo['gold'], ensure_ascii=False, separators=(',', ':'))},
        ])
    messages.append({'role': 'user', 'content': complaint})
    return messages

def train_records(rows):
    return [
        {'prompt': messages_for(row['complaint']), 'completion': [{'role': 'assistant', 'content': json.dumps(row['gold'], ensure_ascii=False, separators=(',', ':'))}]}
        for row in rows
    ]

STATIC_IDS = ('canonical-001', 'canonical-020', 'canonical-030')
training_by_id = {row['case_id']: row for row in training}
static_demos = [training_by_id[case_id] for case_id in STATIC_IDS]
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
training_matrix = vectorizer.fit_transform([row['complaint'] for row in training])

def retrieve(complaint):
    scores = (training_matrix @ vectorizer.transform([complaint]).T).toarray().ravel()
    ranked = scores.argsort()[::-1]
    selected = [training[index] for index in ranked[:DEMO_COUNT]]
    assert len({row['case_id'] for row in selected}) == DEMO_COUNT
    return selected

def retrieved_lookup(rows):
    selected = {row['surface_id']: retrieve(row['complaint']) for row in rows}
    assert all(all(demo['split'] == 'train' for demo in demos) for demos in selected.values())
    return selected

retrieved_validation = retrieved_lookup(validation)
print({'static_demo_ids': list(STATIC_IDS), 'retrieval_training_rows': len(training), 'validation_retrieval_ready': len(retrieved_validation)})

In [ ]:
def rule_output(complaint):
    text = complaint.casefold()
    if any(word in text for word in ('water', 'tap ', 'tanker')):
        domain = 'water_supply'
    elif any(word in text for word in ('garbage', 'waste', 'sewage', 'sanitation', 'bin ', 'sweeper')):
        domain = 'sanitation_and_waste'
    elif any(word in text for word in ('electric', 'power', 'feeder', 'transformer')):
        domain = 'electricity'
    elif any(word in text for word in ('road', 'streetlight', 'street light', 'pothole', 'signal', 'sidewalk', 'pavement', 'parking meter')):
        domain = 'roads_and_streetlights'
    elif any(word in text for word in ('pension', 'certificate', 'benefit', 'welfare', 'application', 'document')):
        domain = 'welfare_or_document_service'
    elif any(word in text for word in ('bus', 'train', 'station', 'platform', 'ticket', 'route ')):
        domain = 'public_transport'
    else:
        domain = 'other'
    if any(word in text for word in ('charged', 'bill ', 'fee ', 'payment')):
        issue = 'overcharging_or_payment_problem'
    elif any(word in text for word in ('wrong', 'incorrect', 'marked completed', 'record ')):
        issue = 'record_or_document_error'
    elif any(word in text for word in ('shouted', 'insulted', 'rude', 'refused', 'mocked', 'ignored')):
        issue = 'staff_conduct'
    elif any(word in text for word in ('live wire', 'sparking', 'unsafe', 'hazard', 'collision', 'needles', 'sharp edges')):
        issue = 'safety_or_health_hazard'
    elif any(word in text for word in ('did not arrive', 'never arrived', 'never came', 'did not come', 'scheduled for', 'was due')):
        issue = 'delay_or_non_arrival'
    elif any(word in text for word in ('broken', 'cracked', 'damaged', 'pothole', 'leaking', 'raised sidewalk', 'leaning')):
        issue = 'damaged_infrastructure'
    elif any(word in text for word in ('no water', 'no power', 'unavailable', 'not working', 'has not worked', 'error for', 'no collection')):
        issue = 'service_outage_or_non_delivery'
    else:
        issue = 'other'
    urgency = 'safety_critical' if issue == 'safety_or_health_hazard' else ('time_sensitive' if any(word in text for word in ('for two days', 'for three days', 'for four days', 'for five days', 'for six days', 'for a week', 'since monday', 'since friday', 'since sunday')) else 'routine')
    return {'service_domain': domain, 'issue_type': issue, 'location': None, 'event_date_or_time': None, 'amount_inr': None, 'service_identifier': None, 'urgency': urgency, 'missing_information': ['exact_location'], 'formal_summary': complaint.strip()}

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=MODEL_DTYPE, bnb_4bit_use_double_quant=True)

def load_base():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, quantization_config=quantization_config, dtype=MODEL_DTYPE, device_map='auto')
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.config.use_cache = True
    return tokenizer, model

def encoded_batch(tokenizer, message_batch):
    kwargs = dict(tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt', padding=True, truncation=True, max_length=2048, enable_thinking=False)
    try:
        return tokenizer.apply_chat_template(message_batch, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking')
        return tokenizer.apply_chat_template(message_batch, **kwargs)

def generate_rows(tokenizer, model, rows, demo_lookup=None):
    outputs = []
    model.eval()
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        messages = [messages_for(row['complaint'], () if demo_lookup is None else demo_lookup[row['surface_id']]) for row in batch]
        inputs = encoded_batch(tokenizer, messages).to(next(model.parameters()).device)
        prompt_lengths = inputs['attention_mask'].sum(dim=1).tolist()
        padded_length = inputs['input_ids'].shape[1]
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, use_cache=True, pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        responses = tokenizer.batch_decode(generated[:, padded_length:], skip_special_tokens=True)
        outputs.extend([{'surface_id': row['surface_id'], 'case_id': row['case_id'], 'response': response.strip(), 'latency_seconds': elapsed / len(batch), 'prompt_tokens': int(prompt_tokens)} for row, response, prompt_tokens in zip(batch, responses, prompt_lengths)])
    return outputs

def prepare_trainable_parameters(model):
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            parameter.data = parameter.data.float()
    assert all(parameter.dtype == torch.float32 for parameter in model.parameters() if parameter.requires_grad)

In [ ]:
def score_record(method, rows, outputs, *, model_name=MODEL_NAME, model_revision=MODEL_REVISION, demos=0, extra=None):
    scores = evaluate_outputs([row['gold'] for row in rows], [item['response'] for item in outputs])
    record = {'method': method, 'model_name': model_name, 'model_revision': model_revision, 'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS}, 'mean_latency_seconds': sum(item['latency_seconds'] for item in outputs) / len(outputs), 'mean_prompt_tokens': sum(item['prompt_tokens'] for item in outputs) / len(outputs), 'demonstration_count': demos, 'scores': scores, 'outputs': outputs}
    if extra:
        record.update(extra)
    with mlflow.start_run(run_name=method) as run:
        record['mlflow_run_id'] = run.info.run_id
        mlflow.log_params({'method': method, 'model_name': model_name or 'none', 'model_revision': model_revision or 'none', 'demonstration_count': demos})
        mlflow.log_metrics({'schema_validity_rate': scores['strict']['schema_validity_rate'], 'mean_latency_seconds': record['mean_latency_seconds'], 'mean_prompt_tokens': record['mean_prompt_tokens']})
        mlflow.log_text(json.dumps(record, indent=2, ensure_ascii=False, default=json_default), 'results.json')
    return record

def train_one(rows, epochs, run_name, save_dir=None):
    tokenizer, base = load_base()
    base.config.use_cache = False
    torch.cuda.reset_peak_memory_stats()
    lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules='all-linear')
    args = SFTConfig(output_dir=f'/kaggle/working/{run_name}_checkpoints', num_train_epochs=epochs, per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', logging_steps=1, save_strategy='no', report_to='none', fp16=False, bf16=False, max_length=MAX_LENGTH, completion_only_loss=True, gradient_checkpointing=True, seed=SEED)
    trainer = SFTTrainer(model=base, args=args, train_dataset=Dataset.from_list(train_records(rows)), processing_class=tokenizer, peft_config=lora_config)
    prepare_trainable_parameters(trainer.model)
    trainable = sum(parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in trainer.model.parameters())
    assert 0 < trainable < total
    started = time.perf_counter()
    train_result = trainer.train()
    training_seconds = time.perf_counter() - started
    losses = [item['loss'] for item in trainer.state.log_history if 'loss' in item]
    assert losses and all(math.isfinite(loss) for loss in losses)
    if save_dir is not None:
        save_dir = Path(save_dir)
        if save_dir.exists():
            shutil.rmtree(save_dir)
        trainer.save_model(save_dir)
        tokenizer.save_pretrained(save_dir)
    metadata = {'run_name': run_name, 'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_rows': len(rows), 'dataset_version': manifest['dataset_version'], 'epochs': epochs, 'configuration': args.to_dict(), 'lora': lora_config.to_dict(), 'losses': losses, 'train_metrics': train_result.metrics, 'training_seconds': training_seconds, 'peak_gpu_memory_mb': torch.cuda.max_memory_allocated() / 1024**2, 'device': torch.cuda.get_device_name(0), 'trainable_parameters': trainable, 'total_parameters': total, 'packages': {name: version(name) for name in ('torch', 'transformers', 'peft', 'trl', 'bitsandbytes', 'datasets', 'mlflow')}}
    with mlflow.start_run(run_name=run_name) as run:
        metadata['mlflow_run_id'] = run.info.run_id
        mlflow.log_params({'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_rows': len(rows), 'epochs': epochs, 'lora_r': 16, 'learning_rate': 2e-4})
        mlflow.log_metrics({'train_loss': train_result.metrics['train_loss'], 'training_seconds': training_seconds, 'peak_gpu_memory_mb': metadata['peak_gpu_memory_mb']})
        mlflow.log_text(json.dumps(metadata, indent=2, default=json_default), 'training_metadata.json')
    return {'trainer': trainer, 'tokenizer': tokenizer, 'metadata': metadata, 'adapter_dir': save_dir}

def validation_record(model, tokenizer, name):
    outputs = generate_rows(tokenizer, model, validation)
    return score_record(name, validation, outputs)

## revision and ablation

This is the last validation-only section. The full revision uses two epochs instead of the recorded one epoch. The 25 percent and 50 percent runs use seeded case-group subsets and the same two-epoch recipe. The 100 percent point is the revision result.

In [ ]:
final_adapter_dir = OUTPUT / 'final_adapter'
revision = train_one(training, REVISION_EPOCHS, 'smollm3-qlora-revision-2-epochs', final_adapter_dir)
revision_validation = validation_record(revision['trainer'].model, revision['tokenizer'], 'revision-validation-in-memory')
save_json(OUTPUT / 'revision_training_metadata.json', revision['metadata'])
save_json(OUTPUT / 'revision_validation_predictions.json', revision_validation)
del revision['trainer'], revision['tokenizer']
gc.collect()
torch.cuda.empty_cache()
reload_tokenizer, reload_base = load_base()
reloaded_revision = PeftModel.from_pretrained(reload_base, final_adapter_dir)
reloaded_validation = validation_record(reloaded_revision, reload_tokenizer, 'revision-validation-reloaded')
assert reloaded_validation['scores']['strict']['schema_validity_rate'] == revision_validation['scores']['strict']['schema_validity_rate']
save_json(OUTPUT / 'revision_validation_reloaded.json', reloaded_validation)
reference = json.loads((ROOT / 'data/validation_results/qlora_validation_predictions.json').read_text(encoding='utf-8'))
reference_scores = reference['scores']['strict']
revision_scores = reloaded_validation['scores']['strict']
print({'previous_one_epoch': {'schema_validity': reference_scores['schema_validity_rate'], 'missing_information_f1': reference_scores['end_to_end_field_metrics']['missing_information_macro_f1']}, 'revision_two_epoch': {'schema_validity': revision_scores['schema_validity_rate'], 'missing_information_f1': revision_scores['end_to_end_field_metrics']['missing_information_macro_f1']}, 'final_candidate': 'revision_two_epoch'})
del reloaded_revision, reload_base, reload_tokenizer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def grouped_subset(rows, fraction):
    groups = defaultdict(list)
    for row in rows:
        groups[row['case_id']].append(row)
    target = round(len(rows) * fraction)
    case_ids = list(groups)
    random.Random(SEED + round(fraction * 100)).shuffle(case_ids)
    chosen = []
    count = 0
    for case_id in case_ids:
        if count + len(groups[case_id]) > target:
            continue
        chosen.append(case_id)
        count += len(groups[case_id])
        if count == target:
            break
    assert count == target
    chosen_set = set(chosen)
    return [row for row in rows if row['case_id'] in chosen_set], chosen

ablation = []
for fraction in (0.25, 0.50):
    subset, case_ids = grouped_subset(training, fraction)
    run = train_one(subset, REVISION_EPOCHS, f'smollm3-qlora-ablation-{int(fraction * 100)}pct', save_dir=None)
    validation_result = validation_record(run['trainer'].model, run['tokenizer'], f'ablation-{int(fraction * 100)}pct-validation')
    ablation.append({'fraction': fraction, 'rows': len(subset), 'case_ids': case_ids, 'training': run['metadata'], 'validation': validation_result})
    del run['trainer'], run['tokenizer']
    gc.collect()
    torch.cuda.empty_cache()

learning_curve = [{'fraction': 0.25, 'rows': ablation[0]['rows'], 'validation': ablation[0]['validation']}, {'fraction': 0.50, 'rows': ablation[1]['rows'], 'validation': ablation[1]['validation']}, {'fraction': 1.00, 'rows': len(training), 'validation': reloaded_validation}]
save_json(OUTPUT / 'ablation_results.json', {'training_recipe': {'epochs': REVISION_EPOCHS, 'seed': SEED, 'learning_rate': 2e-4, 'lora_r': 16, 'lora_alpha': 32}, 'runs': ablation, 'learning_curve': learning_curve})
save_json(OUTPUT / 'frozen_system_manifest.json', {'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_version': manifest['dataset_version'], 'training_rows': len(training), 'validation_rows': len(validation), 'epochs': REVISION_EPOCHS, 'seed': SEED, 'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS}, 'demo_count': DEMO_COUNT, 'retrieval': 'TF-IDF over training rows only', 'adapter_path': str(final_adapter_dir), 'test_loaded': False})
print({'ablation_rows': [run['rows'] for run in ablation], 'learning_curve_saved': True, 'test_loaded': False})

## freeze, then open the held-out slices

The validation comparison and data-size ablation are now saved. The next cell fetches the two held-out files. Do not change any experiment setting after this point.

In [ ]:
FINAL_FILES = ('data/test_cases.jsonl', 'data/external_civic_eval.jsonl')
for relative_path in FINAL_FILES:
    fetch(relative_path)
assert sha256(ROOT / 'data/test_cases.jsonl') == manifest['sha256']['data/test_cases.jsonl']
assert sha256(ROOT / 'data/external_civic_eval.jsonl') == manifest['sha256']['data/external_civic_eval.jsonl']
test_rows = load_jsonl(ROOT / 'data/test_cases.jsonl')
external_rows = load_jsonl(ROOT / 'data/external_civic_eval.jsonl')
for row in test_rows + external_rows:
    row.setdefault('surface_id', row['case_id'])
assert len(test_rows) == 50 and {row['split'] for row in test_rows} == {'test'}
assert len(external_rows) == 20 and {row['split'] for row in external_rows} == {'external_test'}
assert not {row['case_id'] for row in test_rows + external_rows} & {row['case_id'] for row in training + validation}
save_json(OUTPUT / 'frozen_system_manifest.json', {'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_version': manifest['dataset_version'], 'training_rows': len(training), 'validation_rows': len(validation), 'test_rows': len(test_rows), 'external_rows': len(external_rows), 'epochs': REVISION_EPOCHS, 'seed': SEED, 'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS}, 'demo_count': DEMO_COUNT, 'retrieval': 'TF-IDF over training rows only', 'data_hashes': {path: sha256(ROOT / path) for path in ('data/surface_variants.jsonl', 'data/public_training_examples.jsonl', 'data/test_cases.jsonl', 'data/external_civic_eval.jsonl')}, 'adapter_path': str(final_adapter_dir), 'test_loaded': True})
print({'test_rows': len(test_rows), 'external_rows': len(external_rows), 'system_frozen': True})

In [ ]:
def run_prompting_comparison(rows, name):
    retrieval = retrieved_lookup(rows)
    results = {}
    rule_outputs = [{'surface_id': row['surface_id'], 'case_id': row['case_id'], 'response': json.dumps(rule_output(row['complaint']), separators=(',', ':')), 'latency_seconds': 0.0, 'prompt_tokens': 0} for row in rows]
    results['deterministic_rules'] = score_record(f'{name}-deterministic-rules', rows, rule_outputs, model_name=None, model_revision=None)
    tokenizer, base = load_base()
    outputs = generate_rows(tokenizer, base, rows)
    results['zero_shot'] = score_record(f'{name}-zero-shot', rows, outputs, extra={'public_method': 'zero_shot'})
    outputs = generate_rows(tokenizer, base, rows, {row['surface_id']: static_demos for row in rows})
    results['static_few_shot'] = score_record(f'{name}-static-few-shot', rows, outputs, demos=DEMO_COUNT, extra={'public_method': 'static_few_shot'})
    outputs = generate_rows(tokenizer, base, rows, retrieval)
    results['retrieved_few_shot'] = score_record(f'{name}-retrieved-few-shot', rows, outputs, demos=DEMO_COUNT, extra={'retrieved_case_ids': {row['surface_id']: [demo['case_id'] for demo in retrieval[row['surface_id']]] for row in rows}})
    del base, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return results

internal_results = run_prompting_comparison(test_rows, 'internal-test')
external_results = run_prompting_comparison(external_rows, 'external-transfer')
print({'internal_prompting_methods': list(internal_results), 'external_prompting_methods': list(external_results)})

In [ ]:
tokenizer, base = load_base()
final_model = PeftModel.from_pretrained(base, final_adapter_dir)
internal_outputs = generate_rows(tokenizer, final_model, test_rows)
external_outputs = generate_rows(tokenizer, final_model, external_rows)
internal_results['qlora'] = score_record('internal-test-qlora', test_rows, internal_outputs, extra={'adapter_epochs': REVISION_EPOCHS, 'adapter_path': str(final_adapter_dir)})
external_results['qlora'] = score_record('external-transfer-qlora', external_rows, external_outputs, extra={'adapter_epochs': REVISION_EPOCHS, 'adapter_path': str(final_adapter_dir)})
save_json(OUTPUT / 'internal_test_results.json', internal_results)
save_json(OUTPUT / 'external_transfer_results.json', external_results)
with mlflow.start_run(run_name='final-adapter-artifact') as run:
    mlflow.log_params({'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_version': manifest['dataset_version'], 'epochs': REVISION_EPOCHS})
    mlflow.log_artifacts(str(final_adapter_dir), artifact_path='adapter')
    print({'final_adapter_mlflow_run_id': run.info.run_id})
del final_model, base, tokenizer
gc.collect()
torch.cuda.empty_cache()
print({'internal_rows': len(test_rows), 'external_rows': len(external_rows), 'test_predictions_saved': True})

In [ ]:
save_json(OUTPUT / 'run_summary.json', {'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_version': manifest['dataset_version'], 'validation_revision': reloaded_validation['scores'], 'internal_test_schema_validity': {method: record['scores']['strict']['schema_validity_rate'] for method, record in internal_results.items()}, 'external_schema_validity': {method: record['scores']['strict']['schema_validity_rate'] for method, record in external_results.items()}, 'test_predictions_are_final': True, 'external_slice_is_separate': True})
archive = shutil.make_archive('/kaggle/working/civicstruct_final_results', 'zip', root_dir=str(OUTPUT))
print({'archive': archive, 'output_bytes': sum(path.stat().st_size for path in OUTPUT.rglob('*') if path.is_file())})